In [1]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Drive monté")

Mounted at /content/drive
✓ Drive monté


In [2]:
import warnings
import json
import time
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠ No GPU — Runtime → Change runtime type → GPU")

print("\n✓ Imports OK")

Device : cuda
GPU    : NVIDIA A100-SXM4-80GB
VRAM   : 85.1 GB

✓ Imports OK


In [3]:
#cell 2 ── Architecture deux disques ─────────────────────────────────────────────────
# Lecture  : Google Drive (données sources — jamais modifiées)
# Écriture : disque local Colab (/content) — rapide, pas de quota Drive
# Backup   : zip périodique téléchargé localement (cellule finale)

DRIVE_DIR = Path("/content/drive/MyDrive")   # source — lecture seule
BASE_DIR  = Path("/content/aneo_work")       # destination — écriture locale

DATA_DIR      = DRIVE_DIR / "data_4x"        # données brutes sur Drive
OUT_DIR       = BASE_DIR  / "aneo_output"
MODEL_DIR     = BASE_DIR  / "aneo_models"
TENSOR_DIR    = BASE_DIR  / "aneo_tensors"
FIG_DIR       = BASE_DIR  / "aneo_figures"

# Créer tous les dossiers locaux
for d in [OUT_DIR, MODEL_DIR, TENSOR_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print(f"✓ {d}")

# Vérifier accès aux données sources
print(f"\nAccès Drive source :")
print(f"  data_4x existe         : {DATA_DIR.exists()}")
print(f"  aneo_grid_points.csv   : {(DATA_DIR/'aneo_grid_points.csv').exists()}")
csv_files = list(DATA_DIR.glob("*.csv"))
print(f"  CSV files dans data_4x : {len(csv_files)}")

print(f"\n✓ Architecture deux disques OK")
print(f"  Lecture  → {DATA_DIR}")
print(f"  Écriture → {BASE_DIR}")

✓ /content/aneo_work/aneo_output
✓ /content/aneo_work/aneo_models
✓ /content/aneo_work/aneo_tensors
✓ /content/aneo_work/aneo_figures

Accès Drive source :
  data_4x existe         : True
  aneo_grid_points.csv   : True
  CSV files dans data_4x : 32

✓ Architecture deux disques OK
  Lecture  → /content/drive/MyDrive/data_4x
  Écriture → /content/aneo_work


In [5]:
# cell 3── Créer les tenseurs depuis les CSV bruts ───────────────────────────────────
if (TENSOR_DIR / "split.json").exists():
    print("✓ Tenseurs déjà créés — cellule ignorée")
    print(f"  Fichiers .npy : {len(list(TENSOR_DIR.glob('*.npy')))}")
else:
    print("Création des tenseurs depuis les CSV...")

    # ── 1. Charger le fichier grid points (terrain) ───────────────────────────
    print("\nÉtape 1/4 — Chargement grid points...")
    grid = pd.read_csv(DATA_DIR / "aneo_grid_points.csv")
    grid = grid.sort_values(['iy', 'ix']).reset_index(drop=True)
    print(f"  Grid points : {len(grid):,} points")
    print(f"  Colonnes    : {grid.columns.tolist()}")

    NX, NY = 1000, 500
    assert len(grid) == NX * NY, \
        f"Grid points: {len(grid)} ≠ {NX*NY}"

    # Encoder land_use_code
    land_use_vals = sorted(grid['land_use_code'].dropna().unique())
    land_use_map  = {v: i for i, v in enumerate(land_use_vals)}
    grid['land_use_enc'] = grid['land_use_code'].map(
        land_use_map).fillna(0).astype(np.float32)
    lu_mean = float(grid['land_use_enc'].mean())
    lu_std  = float(grid['land_use_enc'].std()) + 1e-8
    grid['land_use_enc_norm'] = (grid['land_use_enc'] - lu_mean) / lu_std

    print(f"  land_use_map : {land_use_map}")
    print(f"  ✓ Grid points chargés")

    # ── 2. Identifier les fichiers d'observations ─────────────────────────────
    print("\nÉtape 2/4 — Identification des fichiers...")
    obs_files = sorted(DATA_DIR.glob("aneo_observations_day_*.csv"))
    print(f"  Fichiers observations : {len(obs_files)}")

    # Colonnes des fichiers observations
    OBS_COLS = [
        'point_id', 'day_index', 'hour_utc',
        'temperature_c', 'air_density_kg_m3',
        'relative_humidity_pct', 'pressure_pa',
        'wind_u_m_s', 'wind_v_m_s', 'wind_w_m_s',
    ]
    TARGET_COLS        = ['wind_u_m_s', 'wind_v_m_s', 'wind_w_m_s']
    NUMERIC_INPUT_COLS = [
        'temperature_c', 'air_density_kg_m3',
        'relative_humidity_pct', 'pressure_pa',
        'wind_u_m_s', 'wind_v_m_s', 'wind_w_m_s',
    ]
    DAY_COL  = 'day_index'
    HOUR_COL = 'hour_utc'
    N_INPUT_CH = len(NUMERIC_INPUT_COLS) + 1  # 7 + 1 land_use = 8

    # ── 3. Premier passage: calculer stats de normalisation ───────────────────
    print("\nÉtape 3/4 — Calcul des stats de normalisation...")
    sum_dict  = {c: 0.0 for c in NUMERIC_INPUT_COLS}
    sum2_dict = {c: 0.0 for c in NUMERIC_INPUT_COLS}
    count     = 0
    days_set  = set()
    hours_set = set()

    for i, f in enumerate(obs_files):
        df = pd.read_csv(f, usecols=OBS_COLS)
        df = df.dropna(subset=[DAY_COL, HOUR_COL])

        for col in NUMERIC_INPUT_COLS:
            vals = df[col].values.astype(np.float64)
            sum_dict[col]  += vals.sum()
            sum2_dict[col] += (vals**2).sum()

        days_set.update(df[DAY_COL].unique())
        hours_set.update(df[HOUR_COL].unique())
        count += len(df)

        if (i+1) % 10 == 0:
            print(f"  {i+1}/{len(obs_files)} fichiers analysés...")

    mean_dict = {c: sum_dict[c] / count for c in sum_dict}
    std_dict  = {
        c: float(np.sqrt(max(sum2_dict[c]/count - mean_dict[c]**2, 0) + 1e-8))
        for c in sum_dict
    }

    days  = sorted([int(d) for d in days_set])
    hours = sorted([int(h) for h in hours_set])

    print(f"\n  Jours     : {days}")
    print(f"  Heures    : {hours}")
    print(f"  Points total analysés : {count:,}")
    print(f"\n  Stats wind_u : mean={mean_dict['wind_u_m_s']:.4f}  "
          f"std={std_dict['wind_u_m_s']:.4f}")
    print(f"  Stats wind_v : mean={mean_dict['wind_v_m_s']:.4f}  "
          f"std={std_dict['wind_v_m_s']:.4f}")
    print(f"  Stats wind_w : mean={mean_dict['wind_w_m_s']:.4f}  "
          f"std={std_dict['wind_w_m_s']:.4f}")

    # Sauvegarder norm stats
    norm_df = pd.DataFrame({
        col: {'mean': mean_dict[col], 'std': std_dict[col]}
        for col in TARGET_COLS
    }).T
    norm_df.to_csv(OUT_DIR / "normalization_stats.csv")
    print(f"  ✓ normalization_stats.csv sauvegardé")

    # ── 4. Deuxième passage: créer les tenseurs ───────────────────────────────
    print(f"\nÉtape 4/4 — Création des tenseurs .npy...")
    n_created = 0
    n_skipped = 0

    # Préparer le grid pour le merge: index par point_id
    grid_indexed = grid.set_index('point_id')

    for i, f in enumerate(obs_files):
        # Charger observations
        df = pd.read_csv(f, usecols=OBS_COLS)
        df = df.dropna(subset=[DAY_COL, HOUR_COL])
        df[DAY_COL]  = df[DAY_COL].astype(int)
        df[HOUR_COL] = df[HOUR_COL].astype(int)

        # Merger avec grid points sur point_id
        df = df.merge(
            grid_indexed[['ix', 'iy', 'land_use_enc_norm']],
            left_on='point_id', right_index=True, how='left'
        )

        # Normaliser colonnes numériques
        for col in NUMERIC_INPUT_COLS:
            df[col + '_norm'] = (
                (df[col] - mean_dict[col]) / std_dict[col]
            ).astype(np.float32)

        for col in TARGET_COLS:
            df[col + '_norm'] = (
                (df[col] - mean_dict[col]) / std_dict[col]
            ).astype(np.float32)

        input_norm_cols  = [c + '_norm' for c in NUMERIC_INPUT_COLS] + \
                           ['land_use_enc_norm']
        target_norm_cols = [c + '_norm' for c in TARGET_COLS]

        # Créer tenseur par snapshot (day, hour)
        for day in df[DAY_COL].unique():
            for hour in df[HOUR_COL].unique():
                snap = df[
                    (df[DAY_COL]  == day) &
                    (df[HOUR_COL] == hour)
                ].sort_values(['iy', 'ix']).reset_index(drop=True)

                x_path = TENSOR_DIR / f"day_{day:03d}_h{hour:02d}_X.npy"
                y_path = TENSOR_DIR / f"day_{day:03d}_h{hour:02d}_Y.npy"

                if x_path.exists():
                    n_created += 1
                    continue

                if len(snap) != NX * NY:
                    print(f"  ⚠ J{day} h{hour}: {len(snap)} points "
                          f"(attendu {NX*NY}) — ignoré")
                    n_skipped += 1
                    continue

                X = snap[input_norm_cols].values.reshape(
                    NY, NX, N_INPUT_CH).astype(np.float32)
                Y = snap[target_norm_cols].values.reshape(
                    NY, NX, len(TARGET_COLS)).astype(np.float32)

                np.save(x_path, X)
                np.save(y_path, Y)
                n_created += 1

        import gc; gc.collect()
        print(f"  [{i+1}/{len(obs_files)}] {f.name} — "
              f"{n_created} snapshots créés")

    print(f"\n  Snapshots créés   : {n_created}")
    print(f"  Snapshots ignorés : {n_skipped}")

    # ── 5. Sauvegarder split.json ─────────────────────────────────────────────
    n_days     = len(days)
    val_days   = [days[i] for i in range(4, n_days, 5)]
    test_days  = [days[i] for i in range(2, n_days, 5)]
    train_days = [d for d in days
                  if d not in val_days and d not in test_days]

    split = {
        "train_days"   : train_days,
        "val_days"     : val_days,
        "test_days"    : test_days,
        "heures"       : list(hours),
        "nx"           : NX,
        "ny"           : NY,
        "n_input"      : N_INPUT_CH,
        "n_target"     : len(TARGET_COLS),
        "input_cols"   : input_norm_cols,
        "target_cols"  : TARGET_COLS,
        "day_col"      : DAY_COL,
        "hour_col"     : HOUR_COL,
        "land_use_map" : land_use_map,
        "lu_mean"      : lu_mean,
        "lu_std"       : lu_std,
        "mean_dict"    : mean_dict,
        "std_dict"     : std_dict,
    }

    with open(TENSOR_DIR / "split.json", "w") as f:
        json.dump(split, f, indent=2)

    print(f"\n  Split:")
    print(f"    Train : {len(train_days)} jours → {train_days}")
    print(f"    Val   : {len(val_days)} jours → {val_days}")
    print(f"    Test  : {len(test_days)} jours → {test_days}")

    npy_files = list(TENSOR_DIR.glob("day_*.npy"))
    total_mb  = sum(f.stat().st_size for f in npy_files) / 1e6
    print(f"\n  Fichiers .npy : {len(npy_files)}")
    print(f"  Taille totale : {total_mb:.0f} MB")
    print(f"\n✓ Pipeline complet terminé → {TENSOR_DIR}")

Création des tenseurs depuis les CSV...
  CSV trouvés : 32

Étape 1/3 — Calcul des stats de normalisation...


ValueError: Usecols do not match columns, columns expected but not found: ['relative_humidity_pct', 'wind_w_m_s', 'day_index', 'wind_u_m_s', 'air_density_kg_m3', 'wind_v_m_s', 'hour_utc', 'temperature_c', 'pressure_pa']

In [8]:
# cell 4── Créer grid_features.npy depuis aneo_grid_points.csv ──────────────────────
if (TENSOR_DIR / "grid_features.npy").exists():
    print("✓ grid_features.npy existe déjà")
else:
    print("Création de grid_features.npy...")

    grid = pd.read_csv(DATA_DIR / "aneo_grid_points.csv")
    grid = grid.sort_values(['iy', 'ix']).reset_index(drop=True)
    print(f"  Colonnes : {grid.columns.tolist()}")
    print(f"  Shape    : {grid.shape}")

    NX_g, NY_g = 1000, 500

    def normalize(arr):
        return (arr - arr.mean()) / (arr.std() + 1e-8)

    alt_grid     = normalize(grid['z_terrain_m'].values).reshape(NY_g, NX_g)
    zrel_grid    = normalize(grid['z_rel_m'].values).reshape(NY_g, NX_g)
    z0_grid      = normalize(grid['z0_m'].values).reshape(NY_g, NX_g)
    slope_grid   = normalize(grid['slope_deg'].values).reshape(NY_g, NX_g)
    aspect_rad   = np.deg2rad(grid['aspect_deg'].values)
    asp_sin_grid = np.sin(aspect_rad).reshape(NY_g, NX_g).astype(np.float32)
    asp_cos_grid = np.cos(aspect_rad).reshape(NY_g, NX_g).astype(np.float32)
    x_grid       = normalize(grid['x_m'].values).reshape(NY_g, NX_g)
    y_grid       = normalize(grid['y_m'].values).reshape(NY_g, NX_g)

    grid_features = np.stack([
        alt_grid, zrel_grid, z0_grid, slope_grid,
        asp_sin_grid, asp_cos_grid, x_grid, y_grid,
    ], axis=-1).astype(np.float32)

    np.save(TENSOR_DIR / "grid_features.npy", grid_features)

    print(f"  grid_features shape : {grid_features.shape}  (NY, NX, 8)")
    print(f"  Taille              : {grid_features.nbytes/1e6:.1f} MB")
    print(f"✓ grid_features.npy sauvegardé")

Création de grid_features.npy...
  Colonnes : ['point_id', 'dataset_name', 'grid_name', 'ix', 'iy', 'x_m', 'y_m', 'z_terrain_m', 'z_rel_m', 'z0_m', 'terrain_type', 'land_use_code', 'slope_deg', 'aspect_deg', 'geom_wkt']
  Shape    : (500000, 15)
  grid_features shape : (500, 1000, 8)  (NY, NX, 8)
  Taille              : 16.0 MB
✓ grid_features.npy sauvegardé


In [9]:
#cell 5
with open(TENSOR_DIR / "split.json", "r") as f:
    split = json.load(f)

TRAIN_DAYS  = split["train_days"]
VAL_DAYS    = split["val_days"]
TEST_DAYS   = split["test_days"]
HEURES      = split["heures"]
NX          = split["nx"]
NY          = split["ny"]
N_INPUT     = split["n_input"]
N_TARGET    = split["n_target"]
INPUT_COLS  = split["input_cols"]
TARGET_COLS = split["target_cols"]

norm_stats = pd.read_csv(OUT_DIR / "normalization_stats.csv", index_col=0)

STD_U = norm_stats.loc["wind_u_m_s", "std"]
STD_V = norm_stats.loc["wind_v_m_s", "std"]
STD_W = norm_stats.loc["wind_w_m_s", "std"]
MU_U  = norm_stats.loc["wind_u_m_s", "mean"]
MU_V  = norm_stats.loc["wind_v_m_s", "mean"]
MU_W  = norm_stats.loc["wind_w_m_s", "mean"]

N_GRID_FEATURES     = 8
N_TEMPORAL_FEATURES = 2
N_INPUT_EFFECTIVE   = N_INPUT + N_GRID_FEATURES + N_TEMPORAL_FEATURES  # 18

print("Configuration :\n")
print(f"  Train : {len(TRAIN_DAYS)} jours → {TRAIN_DAYS}")
print(f"  Val   : {len(VAL_DAYS)} jours → {VAL_DAYS}")
print(f"  Test  : {len(TEST_DAYS)} jours → {TEST_DAYS}")
print(f"\n  Grille : {NX} × {NY}")
print(f"  N_INPUT_EFFECTIVE : {N_INPUT_EFFECTIVE}")
print(f"  N_TARGET          : {N_TARGET}")
print(f"\n  STD u,v,w : {STD_U:.4f}, {STD_V:.4f}, {STD_W:.4f}")
print(f"  MU  u,v,w : {MU_U:.4f}, {MU_V:.4f}, {MU_W:.4f}")
print(f"\n✓ Config OK")

Configuration :

  Train : 18 jours → [1, 2, 4, 6, 7, 9, 11, 12, 14, 16, 17, 19, 21, 22, 24, 26, 27, 29]
  Val   : 6 jours → [5, 10, 15, 20, 25, 30]
  Test  : 6 jours → [3, 8, 13, 18, 23, 28]

  Grille : 1000 × 500
  N_INPUT_EFFECTIVE : 18
  N_TARGET          : 3

  STD u,v,w : 2.0157, 2.1649, 0.0624
  MU  u,v,w : 1.5706, 4.1352, -0.0274

✓ Config OK


In [10]:
#cell 6
class ANEODataset(Dataset):
    """
    18 canaux: 8 originaux + 8 terrain + 2 temporels.
    Pas de rotation (terrain statique).
    """
    def __init__(self, days, heures, tensor_dir, augment=False):
        self.tensor_dir = Path(tensor_dir)
        self.augment    = augment
        self.samples    = []

        for day in days:
            for h in heures:
                x_path = self.tensor_dir / f"day_{day:03d}_h{h:02d}_X.npy"
                y_path = self.tensor_dir / f"day_{day:03d}_h{h:02d}_Y.npy"
                if x_path.exists() and y_path.exists():
                    self.samples.append((x_path, y_path, day, h))

        gf = np.load(self.tensor_dir / "grid_features.npy")
        self.grid_features = torch.from_numpy(
            gf.transpose(2, 0, 1).astype(np.float32))

        print(f"  Dataset : {len(self.samples)} snapshots "
              f"({'aug ON' if augment else 'aug OFF'})  "
              f"{N_INPUT_EFFECTIVE} channels")

    def __len__(self):
        return len(self.samples)

    def augment_sample(self, X, Y):
        X, Y = X.clone(), Y.clone()
        if torch.rand(1) > 0.5:
            scale = torch.FloatTensor(1).uniform_(0.8, 1.2).item()
            X[4:7] *= scale
            Y      *= scale
        if torch.rand(1) > 0.5:
            X[:8] += torch.randn_like(X[:8]) * 0.02
        return X, Y

    def __getitem__(self, idx):
        x_path, y_path, day, h = self.samples[idx]

        X = torch.from_numpy(
            np.load(x_path).astype(np.float32).transpose(2, 0, 1))
        Y = torch.from_numpy(
            np.load(y_path).astype(np.float32).transpose(2, 0, 1))

        if self.augment:
            X, Y = self.augment_sample(X, Y)

        X = torch.cat([X, self.grid_features], dim=0)

        _, H, W = X.shape
        hour_sin = float(np.sin(2 * np.pi * h / 24))
        hour_cos = float(np.cos(2 * np.pi * h / 24))
        sin_ch = torch.full((1, H, W), hour_sin, dtype=torch.float32)
        cos_ch = torch.full((1, H, W), hour_cos, dtype=torch.float32)
        X = torch.cat([X, sin_ch, cos_ch], dim=0)

        return X, Y, day, h

print("✓ ANEODataset défini")

✓ ANEODataset défini


In [11]:
#cell 7
BATCH_SIZE = 1

train_dataset = ANEODataset(
    TRAIN_DAYS + VAL_DAYS, HEURES, TENSOR_DIR, augment=True)
val_dataset   = ANEODataset(
    VAL_DAYS,   HEURES, TENSOR_DIR, augment=False)
test_dataset  = ANEODataset(
    TEST_DAYS,  HEURES, TENSOR_DIR, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=1,
                          shuffle=False, num_workers=2)

print(f"Batch size     : {BATCH_SIZE}")
print(f"Train batches  : {len(train_loader)}")
print(f"Val   batches  : {len(val_loader)}")
print(f"Test  batches  : {len(test_loader)}")

X_b, Y_b, _, _ = next(iter(train_loader))
print(f"\nX : {X_b.shape}  ← doit être [1, 18, 500, 1000]")
print(f"Y : {Y_b.shape}  ← doit être [1,  3, 500, 1000]")

assert X_b.shape[1] == N_INPUT_EFFECTIVE, \
    f"Channels incorrects: {X_b.shape[1]} vs {N_INPUT_EFFECTIVE}"

print(f"\n✓ Dataloaders prêts")

  Dataset : 96 snapshots (aug ON)  18 channels
  Dataset : 24 snapshots (aug OFF)  18 channels
  Dataset : 24 snapshots (aug OFF)  18 channels
Batch size     : 1
Train batches  : 96
Val   batches  : 24
Test  batches  : 24

X : torch.Size([1, 18, 500, 1000])  ← doit être [1, 18, 500, 1000]
Y : torch.Size([1, 3, 500, 1000])  ← doit être [1,  3, 500, 1000]

✓ Dataloaders prêts


In [12]:
#cell8
class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes_x, modes_y):
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.modes_x      = modes_x
        self.modes_y      = modes_y
        scale = 1 / (in_channels * out_channels)
        self.weights1 = nn.Parameter(
            scale * torch.rand(in_channels, out_channels,
                               modes_x, modes_y, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(
            scale * torch.rand(in_channels, out_channels,
                               modes_x, modes_y, dtype=torch.cfloat))

    def compl_mul2d(self, input, weights):
        return torch.einsum("bixy,ioxy->boxy", input, weights)

    def forward(self, x):
        B = x.shape[0]
        x_ft = torch.fft.rfft2(x.float())
        out_ft = torch.zeros(
            B, self.out_channels,
            x.size(-2), x.size(-1)//2+1,
            dtype=torch.cfloat, device=x.device)
        out_ft[:,:,:self.modes_x,:self.modes_y] = \
            self.compl_mul2d(
                x_ft[:,:,:self.modes_x,:self.modes_y], self.weights1)
        out_ft[:,:,-self.modes_x:,:self.modes_y] = \
            self.compl_mul2d(
                x_ft[:,:,-self.modes_x:,:self.modes_y], self.weights2)
        return torch.fft.irfft2(out_ft, s=(x.size(-2), x.size(-1)))


class FNOBlock(nn.Module):
    def __init__(self, channels, modes_x, modes_y, dropout=0.05):
        super().__init__()
        self.spectral = SpectralConv2d(channels, channels, modes_x, modes_y)
        self.local    = nn.Conv2d(channels, channels, kernel_size=1)
        self.norm     = nn.InstanceNorm2d(channels, affine=True)
        self.dropout  = nn.Dropout2d(p=dropout)

    def forward(self, x):
        return F.gelu(self.dropout(
            self.norm(self.spectral(x) + self.local(x))))


class FNO2D(nn.Module):
    """
    Physics-Constrained FNO2D — meilleur modèle validé.
    18 canaux d'entrée (8 atm + 8 terrain + 2 temporels).
    Têtes séparées u/v/w + skip connection globale.
    """
    def __init__(self, in_channels, out_channels,
                 hidden_channels=96, n_blocks=5,
                 modes_x=20, modes_y=20, dropout=0.05):
        super().__init__()

        self.lifting = nn.Sequential(
            nn.Conv2d(in_channels, hidden_channels, kernel_size=1),
            nn.GELU(),
        )
        self.blocks = nn.ModuleList([
            FNOBlock(hidden_channels, modes_x, modes_y, dropout)
            for _ in range(n_blocks)
        ])
        self.skip_proj = nn.Conv2d(
            hidden_channels, hidden_channels, kernel_size=1)

        def make_head(h):
            return nn.Sequential(
                nn.Conv2d(h, h//2, kernel_size=1), nn.GELU(),
                nn.Conv2d(h//2, h//4, kernel_size=1), nn.GELU(),
                nn.Conv2d(h//4, 1,   kernel_size=1),
            )

        self.head_u = make_head(hidden_channels)
        self.head_v = make_head(hidden_channels)
        self.head_w = make_head(hidden_channels)

    def forward(self, x):
        x    = self.lifting(x)
        skip = self.skip_proj(x)
        for block in self.blocks:
            x = block(x)
        x = x + skip
        return torch.cat([
            self.head_u(x),
            self.head_v(x),
            self.head_w(x),
        ], dim=1)


model = FNO2D(
    in_channels     = N_INPUT_EFFECTIVE,
    out_channels    = N_TARGET,
    hidden_channels = 96,
    n_blocks        = 5,
    modes_x         = 20,
    modes_y         = 20,
    dropout         = 0.05,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"FNO2D Model :")
print(f"  in_channels     : {N_INPUT_EFFECTIVE}")
print(f"  hidden_channels : 96")
print(f"  n_blocks        : 5")
print(f"  modes           : 20×20")
print(f"  skip connection : oui")
print(f"  têtes séparées  : u / v / w")
print(f"  paramètres      : {n_params:,}")
print(f"  device          : {device}")
print(f"\n✓ Modèle prêt")

FNO2D Model :
  in_channels     : 18
  hidden_channels : 96
  n_blocks        : 5
  modes           : 20×20
  skip connection : oui
  têtes séparées  : u / v / w
  paramètres      : 36,940,227
  device          : cuda

✓ Modèle prêt


In [13]:
#cell 9
class PhysicsConstrainedLoss(nn.Module):
    def __init__(self, lambda_mass=0.3, lambda_bc=0.05,
                 lambda_smooth=0.01, lambda_ri=0.02, dx=50.0):
        super().__init__()
        self.lambda_mass   = lambda_mass
        self.lambda_bc     = lambda_bc
        self.lambda_smooth = lambda_smooth
        self.lambda_ri     = lambda_ri
        self.dx            = dx

    def forward(self, pred, target, rho, phys_weight=1.0):
        L_data = F.mse_loss(pred, target)

        u_pred = pred[:, 0:1, :, :]
        v_pred = pred[:, 1:2, :, :]

        rho_u    = rho * u_pred
        rho_v    = rho * v_pred
        drhou_dx = (rho_u[:,:,:,2:] - rho_u[:,:,:,:-2]) / (2*self.dx)
        drhov_dy = (rho_v[:,:,2:,:] - rho_v[:,:,:-2,:]) / (2*self.dx)
        min_y    = min(drhou_dx.shape[2], drhov_dy.shape[2])
        min_x    = min(drhou_dx.shape[3], drhov_dy.shape[3])
        div      = (drhou_dx[:,:,:min_y,:min_x] +
                    drhov_dy[:,:,:min_y,:min_x])
        L_mass   = (div**2).mean()

        w_pred = pred[:, 2:3, :, :]
        L_bc   = (w_pred[:,:,0,:]**2).mean()

        grad_u_x = pred[:,0:1,:,1:] - pred[:,0:1,:,:-1]
        grad_u_y = pred[:,0:1,1:,:] - pred[:,0:1,:-1,:]
        grad_v_x = pred[:,1:2,:,1:] - pred[:,1:2,:,:-1]
        grad_v_y = pred[:,1:2,1:,:] - pred[:,1:2,:-1,:]
        L_smooth = ((grad_u_x**2).mean() + (grad_u_y**2).mean() +
                    (grad_v_x**2).mean() + (grad_v_y**2).mean())

        du_dx  = (u_pred[:,:,:,1:] - u_pred[:,:,:,:-1]) / self.dx
        dv_dy  = (v_pred[:,:,1:,:] - v_pred[:,:,:-1,:]) / self.dx
        L_ri   = (F.relu(du_dx**2-1.0).mean() +
                  F.relu(dv_dy**2-1.0).mean())

        phys_term = (self.lambda_mass   * L_mass +
                     self.lambda_bc     * L_bc +
                     self.lambda_smooth * L_smooth +
                     self.lambda_ri     * L_ri)

        return L_data + phys_weight * phys_term, L_data, L_mass, L_bc


criterion = PhysicsConstrainedLoss(
    lambda_mass=0.3, lambda_bc=0.05,
    lambda_smooth=0.01, lambda_ri=0.02, dx=50.0)

print("✓ Physics loss prête")

✓ Physics loss prête


In [14]:
#cell 10
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr           = 3e-4,
    weight_decay = 1e-4,
    betas        = (0.9, 0.999),
)

N_EPOCHS      = 150
WARMUP_EPOCHS = 15
EARLY_STOP    = 30
ACCUM_STEPS   = 4

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=1e-6)

scaler = GradScaler('cuda')

print(f"  N_EPOCHS      : {N_EPOCHS}")
print(f"  WARMUP_EPOCHS : {WARMUP_EPOCHS}")
print(f"  EARLY_STOP    : {EARLY_STOP}")
print(f"  ACCUM_STEPS   : {ACCUM_STEPS}")
print(f"\n✓ Optimizer prêt")

  N_EPOCHS      : 150
  WARMUP_EPOCHS : 15
  EARLY_STOP    : 30
  ACCUM_STEPS   : 4

✓ Optimizer prêt


In [15]:
#cell 11
model.train()
X_b, Y_b, _, _ = next(iter(train_loader))
X_b, Y_b = X_b.to(device), Y_b.to(device)
rho = X_b[:, 1:2, :, :]

with autocast('cuda', dtype=torch.float16):
    pred = model(X_b)
    loss, l_data, l_mass, l_bc = criterion(
        pred, Y_b, rho, phys_weight=0.0)

assert pred.shape == Y_b.shape, \
    f"Shape mismatch: pred={pred.shape} vs {Y_b.shape}"

scaler.scale(loss).backward()
optimizer.zero_grad()

total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"pred shape    : {pred.shape} ✓")
print(f"L_data        : {l_data.item():.6f}")
print(f"Paramètres    : {sum(p.numel() for p in model.parameters()):,}")
print(f"GPU total     : {total:.1f} GB")
print(f"GPU alloué    : {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU réservé   : {torch.cuda.memory_reserved()/1e9:.2f} GB")
print(f"GPU libre     : {(total-torch.cuda.memory_reserved()/1e9):.2f} GB")
torch.cuda.empty_cache()
print(f"\n✓ OOM check OK")

pred shape    : torch.Size([1, 3, 500, 1000]) ✓
L_data        : 0.912032
Paramètres    : 36,940,227
GPU total     : 85.1 GB
GPU alloué    : 0.36 GB
GPU réservé   : 5.87 GB
GPU libre     : 79.23 GB

✓ OOM check OK


In [16]:
#cell 12
def physics_warmup(epoch, warmup_epochs):
    if warmup_epochs <= 0:
        return 1.0
    return min(1.0, (epoch + 1) / warmup_epochs)


def compute_r2_per_channel(pred, true):
    pred_np = pred.detach().float().cpu().numpy()
    true_np = true.detach().float().cpu().numpy()
    scores  = []
    for c in range(pred_np.shape[1]):
        y_true = true_np[:, c].flatten()
        y_pred = pred_np[:, c].flatten()
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - y_true.mean()) ** 2)
        scores.append(1 - ss_res / (ss_tot + 1e-8))
    return np.mean(scores), scores


def train_epoch(epoch):
    model.train()
    train_loss = 0.0
    phys_w     = physics_warmup(epoch, WARMUP_EPOCHS)
    optimizer.zero_grad()

    for step, (X_b, Y_b, _, _) in enumerate(train_loader):
        X_b, Y_b = X_b.to(device), Y_b.to(device)
        rho = X_b[:, 1:2, :, :]

        with autocast('cuda', dtype=torch.float16):
            pred = model(X_b)
            loss, l_data, l_mass, l_bc = criterion(
                pred, Y_b, rho, phys_weight=phys_w)
            loss_scaled = loss / ACCUM_STEPS

        loss_scaled.backward()

        if (step+1) % ACCUM_STEPS == 0 or (step+1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        train_loss += loss.item()

    return train_loss / len(train_loader)


def validate_epoch(epoch):
    model.eval()
    val_loss      = 0.0
    r2_means      = []
    r2_per_ch_sum = np.zeros(3)
    phys_w        = physics_warmup(epoch, WARMUP_EPOCHS)

    with torch.no_grad():
        for X_b, Y_b, _, _ in val_loader:
            X_b, Y_b = X_b.to(device), Y_b.to(device)
            rho = X_b[:, 1:2, :, :]

            with autocast('cuda', dtype=torch.float16):
                pred = model(X_b)
                loss, _, _, _ = criterion(
                    pred, Y_b, rho, phys_weight=phys_w)

            val_loss += loss.item()
            r2_mean, r2_ch = compute_r2_per_channel(pred, Y_b)
            r2_means.append(r2_mean)
            r2_per_ch_sum += np.array(r2_ch)

    n = len(val_loader)
    return val_loss / n, np.mean(r2_means), (r2_per_ch_sum / n)


print("✓ Fonctions définies")

✓ Fonctions définies


In [17]:
#cell 13
history = {
    'train_loss': [], 'val_loss': [], 'r2': [],
    'r2_u': [], 'r2_v': [], 'r2_w': [],
    'lr': [], 'phys_weight': []
}

best_val_loss    = float('inf')
best_epoch       = 0
best_r2          = -np.inf
patience_counter = 0

print("="*88)
print("STARTING FNO2D TRAINING (18ch — terrain + temporal)")
print("="*88)
print(f"{'Ep':>4} | {'Train':>9} | {'Val':>9} | {'R²avg':>7} | "
      f"{'R²_u':>6} | {'R²_v':>6} | {'R²_w':>6} | "
      f"{'φ_w':>5} | {'LR':>8} | Status")
print("-"*88)

try:
    for epoch in range(N_EPOCHS):
        tr_loss            = train_epoch(epoch)
        va_loss, r2, r2_ch = validate_epoch(epoch)
        scheduler.step()

        lr     = optimizer.param_groups[0]['lr']
        phys_w = physics_warmup(epoch, WARMUP_EPOCHS)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['r2'].append(r2)
        history['r2_u'].append(r2_ch[0])
        history['r2_v'].append(r2_ch[1])
        history['r2_w'].append(r2_ch[2])
        history['lr'].append(lr)
        history['phys_weight'].append(phys_w)

        print(f"{epoch+1:>4} | {tr_loss:>9.5f} | {va_loss:>9.5f} | "
              f"{r2:>7.4f} | {r2_ch[0]:>6.4f} | {r2_ch[1]:>6.4f} | "
              f"{r2_ch[2]:>6.4f} | {phys_w:>5.2f} | {lr:>8.2e} | ", end="")

        if va_loss < best_val_loss:
            best_val_loss    = va_loss
            best_epoch       = epoch + 1
            best_r2          = r2
            patience_counter = 0
            torch.save({
                'epoch'               : epoch + 1,
                'model_state_dict'    : model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss'            : va_loss,
                'r2'                  : r2,
                'r2_u'                : r2_ch[0],
                'r2_v'                : r2_ch[1],
                'r2_w'                : r2_ch[2],
                'history'             : history,
            }, MODEL_DIR / 'best_fno2d_model.pth')
            print("✓ BEST")
        else:
            patience_counter += 1
            print(f"(patience {patience_counter}/{EARLY_STOP})"
                  if patience_counter % 5 == 0 else "")

        if patience_counter >= EARLY_STOP:
            print(f"\n⚠ Early stopping à l'époque {epoch+1}")
            break

except KeyboardInterrupt:
    print("\n⚠ Entraînement interrompu")
except Exception as e:
    print(f"\n✗ Échec: {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()

print("\n" + "="*88)
print("ENTRAÎNEMENT TERMINÉ")
print(f"Meilleure époque : {best_epoch}  "
      f"val_loss : {best_val_loss:.6f}  "
      f"R² : {best_r2:.4f}")
print("="*88)

pd.DataFrame(history).to_csv(
    OUT_DIR / 'fno2d_training_history.csv', index=False)
print(f"✓ Historique sauvegardé")

STARTING FNO2D TRAINING (18ch — terrain + temporal)
  Ep |     Train |       Val |   R²avg |   R²_u |   R²_v |   R²_w |   φ_w |       LR | Status
----------------------------------------------------------------------------------------
   1 |   0.87705 |   0.59311 | -6.0440 | -15.9708 | -2.2232 | 0.0620 |  0.07 | 3.00e-04 | ✓ BEST
   2 |   0.77405 |   0.52880 | -6.1778 | -16.6444 | -2.2311 | 0.3420 |  0.13 | 3.00e-04 | ✓ BEST
   3 |   0.66802 |   0.49906 | -6.1944 | -16.8084 | -2.1787 | 0.4040 |  0.20 | 3.00e-04 | ✓ BEST
   4 |   0.58408 |   0.44601 | -5.5108 | -14.9923 | -2.0448 | 0.5048 |  0.27 | 2.99e-04 | ✓ BEST
   5 |   0.48141 |   0.37992 | -4.5702 | -12.5628 | -1.8013 | 0.6534 |  0.33 | 2.99e-04 | ✓ BEST
   6 |   0.39415 |   0.30489 | -3.3043 | -9.2904 | -1.3477 | 0.7253 |  0.40 | 2.99e-04 | ✓ BEST
   7 |   0.29466 |   0.21599 | -1.6078 | -4.9463 | -0.6370 | 0.7599 |  0.47 | 2.98e-04 | ✓ BEST
   8 |   0.21317 |   0.16948 | -0.6762 | -2.6767 | -0.1033 | 0.7513 |  0.53 | 2.98e-04 |

In [18]:
#cell 14
ckpt = torch.load(MODEL_DIR / "best_fno2d_model.pth",
                  map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Chargé époque {ckpt['epoch']}  "
      f"val_loss={ckpt['val_loss']:.6f}  "
      f"R²avg={ckpt['r2']:.4f}")

results = []
with torch.no_grad():
    for X_b, Y_b, day, hour in test_loader:
        day, hour = day.item(), hour.item()
        with autocast('cuda', dtype=torch.float16):
            pred = model(X_b.to(device))
        pred_n = pred[0].float().cpu().numpy()
        true_n = Y_b[0].numpy()

        u_p = pred_n[0]*STD_U+MU_U; v_p = pred_n[1]*STD_V+MU_V
        w_p = pred_n[2]*STD_W+MU_W
        u_t = true_n[0]*STD_U+MU_U; v_t = true_n[1]*STD_V+MU_V
        w_t = true_n[2]*STD_W+MU_W

        mae_u  = np.mean(np.abs(u_p-u_t))
        mae_v  = np.mean(np.abs(v_p-v_t))
        mae_w  = np.mean(np.abs(w_p-w_t))
        rmse_u = np.sqrt(np.mean((u_p-u_t)**2))
        rmse_v = np.sqrt(np.mean((v_p-v_t)**2))
        rmse_w = np.sqrt(np.mean((w_p-w_t)**2))
        r2_u   = 1-np.sum((u_p-u_t)**2)/(np.sum((u_t-u_t.mean())**2)+1e-9)
        r2_v   = 1-np.sum((v_p-v_t)**2)/(np.sum((v_t-v_t.mean())**2)+1e-9)
        r2_w   = 1-np.sum((w_p-w_t)**2)/(np.sum((w_t-w_t.mean())**2)+1e-9)
        bias_u = np.mean(u_p-u_t)
        bias_v = np.mean(v_p-v_t)
        bias_w = np.mean(w_p-w_t)

        ws_p    = np.sqrt(u_p**2+v_p**2)
        ws_t    = np.sqrt(u_t**2+v_t**2)
        mae_ws  = np.mean(np.abs(ws_p-ws_t))
        dir_p   = np.degrees(np.arctan2(v_p,u_p))%360
        dir_t   = np.degrees(np.arctan2(v_t,u_t))%360
        dir_err = np.minimum(np.abs(dir_p-dir_t),360-np.abs(dir_p-dir_t))
        mae_dir = np.mean(dir_err)

        acc_u_05 = (np.abs(u_p-u_t)<0.5).mean()*100
        acc_u_10 = (np.abs(u_p-u_t)<1.0).mean()*100
        acc_v_05 = (np.abs(v_p-v_t)<0.5).mean()*100
        acc_v_10 = (np.abs(v_p-v_t)<1.0).mean()*100
        acc_w_01 = (np.abs(w_p-w_t)<0.1).mean()*100

        results.append(dict(
            day=day, hour=hour,
            mae_u=mae_u, mae_v=mae_v, mae_w=mae_w,
            rmse_u=rmse_u, rmse_v=rmse_v, rmse_w=rmse_w,
            r2_u=r2_u, r2_v=r2_v, r2_w=r2_w,
            bias_u=bias_u, bias_v=bias_v, bias_w=bias_w,
            mae_ws=mae_ws, mae_dir=mae_dir,
            acc_u_05=acc_u_05, acc_u_10=acc_u_10,
            acc_v_05=acc_v_05, acc_v_10=acc_v_10,
            acc_w_01=acc_w_01,
        ))

        print(f"  J{day:3d} h{hour:02d} | "
              f"MAE u={mae_u:.4f} v={mae_v:.4f} w={mae_w:.4f} | "
              f"R² u={r2_u:.4f} v={r2_v:.4f} w={r2_w:.4f} | "
              f"Acc_u={acc_u_05:.1f}% Acc_v={acc_v_05:.1f}%")

df_r = pd.DataFrame(results)
df_r.to_csv(OUT_DIR / 'fno2d_evaluation_results.csv', index=False)

print(f"\n{'='*72}")
print(f"TEST SET METRICS (moyenne sur {len(df_r)} snapshots):")
print(f"{'='*72}")
for col in ['mae_u','mae_v','mae_w','rmse_u','rmse_v','rmse_w',
            'r2_u','r2_v','r2_w','bias_u','bias_v','bias_w',
            'mae_ws','mae_dir',
            'acc_u_05','acc_u_10','acc_v_05','acc_v_10','acc_w_01']:
    print(f"  {col:12s}: {df_r[col].mean():.4f}")
print('='*72)
print(f"\n✓ Résultats sauvegardés")

Chargé époque 127  val_loss=0.029427  R²avg=0.8913
  J  3 h00 | MAE u=0.1331 v=0.1430 w=0.0072 | R² u=0.8054 v=0.9909 w=0.9657 | Acc_u=98.8% Acc_v=97.9%
  J  3 h06 | MAE u=0.1510 v=0.1814 w=0.0078 | R² u=0.9026 v=0.9863 w=0.9492 | Acc_u=99.0% Acc_v=97.1%
  J  3 h12 | MAE u=0.1621 v=0.2514 w=0.0067 | R² u=0.8394 v=0.9838 w=0.9771 | Acc_u=98.3% Acc_v=90.5%
  J  3 h18 | MAE u=0.1568 v=0.1466 w=0.0061 | R² u=0.7947 v=0.9906 w=0.9762 | Acc_u=98.6% Acc_v=98.1%
  J  8 h00 | MAE u=0.2085 v=0.1324 w=0.0075 | R² u=0.9626 v=0.9885 w=0.9788 | Acc_u=97.1% Acc_v=99.3%
  J  8 h06 | MAE u=0.1766 v=0.1587 w=0.0073 | R² u=0.9563 v=0.9866 w=0.9786 | Acc_u=99.0% Acc_v=98.0%
  J  8 h12 | MAE u=0.1940 v=0.1653 w=0.0066 | R² u=0.9773 v=0.9886 w=0.9870 | Acc_u=94.9% Acc_v=97.5%
  J  8 h18 | MAE u=0.1871 v=0.1222 w=0.0063 | R² u=0.9757 v=0.9855 w=0.9840 | Acc_u=95.3% Acc_v=99.2%
  J 13 h00 | MAE u=0.1548 v=0.1612 w=0.0068 | R² u=0.9501 v=0.9874 w=0.9805 | Acc_u=98.8% Acc_v=97.8%
  J 13 h06 | MAE u=0.1514 v=0.1

In [19]:
#call 15 ── Backup périodique vers ton disque local ───────────────────────────────────
# Zippe tout BASE_DIR et télécharge automatiquement toutes les 5 minutes.
# Lancer cette cellule EN DERNIER — elle tourne en boucle infinie.
# Pour arrêter: interrompre le kernel (bouton stop).

import time
import shutil
from google.colab import files

def periodic_zip_download(base_dir=BASE_DIR, interval=300):
    """
    Zippe BASE_DIR et télécharge toutes les `interval` secondes.
    Contient: modèles, tenseurs, outputs, figures.
    """
    iteration = 0
    print(f"Backup démarré — toutes les {interval//60} minutes")
    print(f"  Source : {base_dir}")
    print(f"  Ctrl+C ou bouton stop pour arrêter\n")

    while True:
        time.sleep(interval)
        iteration += 1

        zip_path = f"/content/aneo_backup_{iteration}"
        print(f"[{iteration}] Création du zip...", end=" ")

        shutil.make_archive(zip_path, 'zip', base_dir)
        size_mb = Path(zip_path + ".zip").stat().st_size / 1e6
        print(f"{size_mb:.0f} MB — téléchargement...")

        files.download(zip_path + ".zip")
        print(f"✓ aneo_backup_{iteration}.zip téléchargé")

        # Nettoyer le zip local pour économiser l'espace
        Path(zip_path + ".zip").unlink()

periodic_zip_download(base_dir=BASE_DIR, interval=300)

Backup démarré — toutes les 5 minutes
  Source : /content/aneo_work
  Ctrl+C ou bouton stop pour arrêter

[1] Création du zip... 3124 MB — téléchargement...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ aneo_backup_1.zip téléchargé
[2] Création du zip... 3124 MB — téléchargement...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ aneo_backup_2.zip téléchargé


KeyboardInterrupt: 

In [7]:
# cell 3── Créer les tenseurs depuis les CSV bruts ───────────────────────────────────
if (TENSOR_DIR / "split.json").exists():
    print("✓ Tenseurs déjà créés — cellule ignorée")
    print(f"  Fichiers .npy : {len(list(TENSOR_DIR.glob('*.npy')))}")
else:
    print("Création des tenseurs depuis les CSV...")

    # ── 1. Charger le fichier grid points (terrain) ───────────────────────────
    print("\nÉtape 1/4 — Chargement grid points...")
    grid = pd.read_csv(DATA_DIR / "aneo_grid_points.csv")
    grid = grid.sort_values(['iy', 'ix']).reset_index(drop=True)
    print(f"  Grid points : {len(grid):,} points")
    print(f"  Colonnes    : {grid.columns.tolist()}")

    NX, NY = 1000, 500
    assert len(grid) == NX * NY, \
        f"Grid points: {len(grid)} ≠ {NX*NY}"

    # Encoder land_use_code
    land_use_vals = sorted(grid['land_use_code'].dropna().unique())
    land_use_map  = {v: i for i, v in enumerate(land_use_vals)}
    grid['land_use_enc'] = grid['land_use_code'].map(
        land_use_map).fillna(0).astype(np.float32)
    lu_mean = float(grid['land_use_enc'].mean())
    lu_std  = float(grid['land_use_enc'].std()) + 1e-8
    grid['land_use_enc_norm'] = (grid['land_use_enc'] - lu_mean) / lu_std

    print(f"  land_use_map : {land_use_map}")
    print(f"  ✓ Grid points chargés")

    # ── 2. Identifier les fichiers d'observations ─────────────────────────────
    print("\nÉtape 2/4 — Identification des fichiers...")
    obs_files = sorted(DATA_DIR.glob("aneo_observations_day_*.csv"))
    print(f"  Fichiers observations : {len(obs_files)}")

    # Colonnes des fichiers observations
    OBS_COLS = [
        'point_id', 'day_index', 'hour_utc',
        'temperature_c', 'air_density_kg_m3',
        'relative_humidity_pct', 'pressure_pa',
        'wind_u_m_s', 'wind_v_m_s', 'wind_w_m_s',
    ]
    TARGET_COLS        = ['wind_u_m_s', 'wind_v_m_s', 'wind_w_m_s']
    NUMERIC_INPUT_COLS = [
        'temperature_c', 'air_density_kg_m3',
        'relative_humidity_pct', 'pressure_pa',
        'wind_u_m_s', 'wind_v_m_s', 'wind_w_m_s',
    ]
    DAY_COL  = 'day_index'
    HOUR_COL = 'hour_utc'
    N_INPUT_CH = len(NUMERIC_INPUT_COLS) + 1  # 7 + 1 land_use = 8

    # ── 3. Premier passage: calculer stats de normalisation ───────────────────
    print("\nÉtape 3/4 — Calcul des stats de normalisation...")
    sum_dict  = {c: 0.0 for c in NUMERIC_INPUT_COLS}
    sum2_dict = {c: 0.0 for c in NUMERIC_INPUT_COLS}
    count     = 0
    days_set  = set()
    hours_set = set()

    for i, f in enumerate(obs_files):
        df = pd.read_csv(f, usecols=OBS_COLS)
        df = df.dropna(subset=[DAY_COL, HOUR_COL])

        for col in NUMERIC_INPUT_COLS:
            vals = df[col].values.astype(np.float64)
            sum_dict[col]  += vals.sum()
            sum2_dict[col] += (vals**2).sum()

        days_set.update(df[DAY_COL].unique())
        hours_set.update(df[HOUR_COL].unique())
        count += len(df)

        if (i+1) % 10 == 0:
            print(f"  {i+1}/{len(obs_files)} fichiers analysés...")

    mean_dict = {c: sum_dict[c] / count for c in sum_dict}
    std_dict  = {
        c: float(np.sqrt(max(sum2_dict[c]/count - mean_dict[c]**2, 0) + 1e-8))
        for c in sum_dict
    }

    days  = sorted([int(d) for d in days_set])
    hours = sorted([int(h) for h in hours_set])

    print(f"\n  Jours     : {days}")
    print(f"  Heures    : {hours}")
    print(f"  Points total analysés : {count:,}")
    print(f"\n  Stats wind_u : mean={mean_dict['wind_u_m_s']:.4f}  "
          f"std={std_dict['wind_u_m_s']:.4f}")
    print(f"  Stats wind_v : mean={mean_dict['wind_v_m_s']:.4f}  "
          f"std={std_dict['wind_v_m_s']:.4f}")
    print(f"  Stats wind_w : mean={mean_dict['wind_w_m_s']:.4f}  "
          f"std={std_dict['wind_w_m_s']:.4f}")

    # Sauvegarder norm stats
    norm_df = pd.DataFrame({
        col: {'mean': mean_dict[col], 'std': std_dict[col]}
        for col in TARGET_COLS
    }).T
    norm_df.to_csv(OUT_DIR / "normalization_stats.csv")
    print(f"  ✓ normalization_stats.csv sauvegardé")

    # ── 4. Deuxième passage: créer les tenseurs ───────────────────────────────
    print(f"\nÉtape 4/4 — Création des tenseurs .npy...")
    n_created = 0
    n_skipped = 0

    # Préparer le grid pour le merge: index par point_id
    grid_indexed = grid.set_index('point_id')

    for i, f in enumerate(obs_files):
        # Charger observations
        df = pd.read_csv(f, usecols=OBS_COLS)
        df = df.dropna(subset=[DAY_COL, HOUR_COL])
        df[DAY_COL]  = df[DAY_COL].astype(int)
        df[HOUR_COL] = df[HOUR_COL].astype(int)

        # Merger avec grid points sur point_id
        df = df.merge(
            grid_indexed[['ix', 'iy', 'land_use_enc_norm']],
            left_on='point_id', right_index=True, how='left'
        )

        # Normaliser colonnes numériques
        for col in NUMERIC_INPUT_COLS:
            df[col + '_norm'] = (
                (df[col] - mean_dict[col]) / std_dict[col]
            ).astype(np.float32)

        for col in TARGET_COLS:
            df[col + '_norm'] = (
                (df[col] - mean_dict[col]) / std_dict[col]
            ).astype(np.float32)

        input_norm_cols  = [c + '_norm' for c in NUMERIC_INPUT_COLS] + \
                           ['land_use_enc_norm']
        target_norm_cols = [c + '_norm' for c in TARGET_COLS]

        # Créer tenseur par snapshot (day, hour)
        for day in df[DAY_COL].unique():
            for hour in df[HOUR_COL].unique():
                snap = df[
                    (df[DAY_COL]  == day) &
                    (df[HOUR_COL] == hour)
                ].sort_values(['iy', 'ix']).reset_index(drop=True)

                x_path = TENSOR_DIR / f"day_{day:03d}_h{hour:02d}_X.npy"
                y_path = TENSOR_DIR / f"day_{day:03d}_h{hour:02d}_Y.npy"

                if x_path.exists():
                    n_created += 1
                    continue

                if len(snap) != NX * NY:
                    print(f"  ⚠ J{day} h{hour}: {len(snap)} points "
                          f"(attendu {NX*NY}) — ignoré")
                    n_skipped += 1
                    continue

                X = snap[input_norm_cols].values.reshape(
                    NY, NX, N_INPUT_CH).astype(np.float32)
                Y = snap[target_norm_cols].values.reshape(
                    NY, NX, len(TARGET_COLS)).astype(np.float32)

                np.save(x_path, X)
                np.save(y_path, Y)
                n_created += 1

        import gc; gc.collect()
        print(f"  [{i+1}/{len(obs_files)}] {f.name} — "
              f"{n_created} snapshots créés")

    print(f"\n  Snapshots créés   : {n_created}")
    print(f"  Snapshots ignorés : {n_skipped}")

    # ── 5. Sauvegarder split.json ─────────────────────────────────────────────
    n_days     = len(days)
    val_days   = [days[i] for i in range(4, n_days, 5)]
    test_days  = [days[i] for i in range(2, n_days, 5)]
    train_days = [d for d in days
                  if d not in val_days and d not in test_days]

    split = {
        "train_days"   : train_days,
        "val_days"     : val_days,
        "test_days"    : test_days,
        "heures"       : list(hours),
        "nx"           : NX,
        "ny"           : NY,
        "n_input"      : N_INPUT_CH,
        "n_target"     : len(TARGET_COLS),
        "input_cols"   : input_norm_cols,
        "target_cols"  : TARGET_COLS,
        "day_col"      : DAY_COL,
        "hour_col"     : HOUR_COL,
        "land_use_map" : land_use_map,
        "lu_mean"      : lu_mean,
        "lu_std"       : lu_std,
        "mean_dict"    : mean_dict,
        "std_dict"     : std_dict,
    }

    with open(TENSOR_DIR / "split.json", "w") as f:
        json.dump(split, f, indent=2)

    print(f"\n  Split:")
    print(f"    Train : {len(train_days)} jours → {train_days}")
    print(f"    Val   : {len(val_days)} jours → {val_days}")
    print(f"    Test  : {len(test_days)} jours → {test_days}")

    npy_files = list(TENSOR_DIR.glob("day_*.npy"))
    total_mb  = sum(f.stat().st_size for f in npy_files) / 1e6
    print(f"\n  Fichiers .npy : {len(npy_files)}")
    print(f"  Taille totale : {total_mb:.0f} MB")
    print(f"\n✓ Pipeline complet terminé → {TENSOR_DIR}")

Création des tenseurs depuis les CSV...

Étape 1/4 — Chargement grid points...
  Grid points : 500,000 points
  Colonnes    : ['point_id', 'dataset_name', 'grid_name', 'ix', 'iy', 'x_m', 'y_m', 'z_terrain_m', 'z_rel_m', 'z0_m', 'terrain_type', 'land_use_code', 'slope_deg', 'aspect_deg', 'geom_wkt']
  land_use_map : {'AGRICULTURE': 0, 'FOREST': 1, 'MOUNTAIN': 2, 'PLAIN': 3, 'URBAN_DENSE': 4, 'URBAN_MEDIUM': 5, 'WATER': 6}
  ✓ Grid points chargés

Étape 2/4 — Identification des fichiers...
  Fichiers observations : 30

Étape 3/4 — Calcul des stats de normalisation...
  10/30 fichiers analysés...
  20/30 fichiers analysés...
  30/30 fichiers analysés...

  Jours     : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
  Heures    : [0, 6, 12, 18]
  Points total analysés : 60,000,000

  Stats wind_u : mean=1.5706  std=2.0157
  Stats wind_v : mean=4.1352  std=2.1649
  Stats wind_w : mean=-0.0274  std=0.0624
  ✓ normalization_stats